# 第20章 金融智能体（AI Agents）—— 配套代码

对应正文 `book/part4/20-ai-agents.md`。本 notebook **从零手写**一个最小可用的 ReAct 智能体，
工具用 `fds` 在内置数据上计算，**离线可跑**；真实 LLM API 部分用 try/except 守卫，无 key 时自动回退到本地规划器。

> 运行前请先生成内置数据：`uv run python scripts/make_sample_data.py`。

In [ ]:
# === 在线运行引导（Google Colab）。本地 / Binder 会自动跳过 ===
import sys

if "google.colab" in sys.modules:
    import os
    import subprocess

    if not os.path.isdir("src/fds"):  # 尚未进入仓库
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/albertandking/financial-data-science", "fds_book"],
                       check=True)
        os.chdir("fds_book")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)
        print("已就绪：仓库已克隆、fds 已安装、内置数据随仓库一并克隆。")

## 1. 准备：工具（本地、离线）

智能体的「手脚」是**工具（tool）**——一组可调用的函数。这里用 `fds` 在内置数据上实现几个金融工具。

In [ ]:
import json
from fds import (load_sample_prices, daily_returns, set_chinese_font,
                 annualized_return, annualized_volatility, sharpe_ratio, max_drawdown)

set_chinese_font()
_PRICES = load_sample_prices()
_RETS = daily_returns(_PRICES)
TICKERS = list(_PRICES.columns)

def tool_list_stocks() -> list:
    """列出可用股票代码。"""
    return TICKERS

def tool_metrics(stock: str) -> dict:
    """返回某只股票的风险收益指标。"""
    if stock not in _RETS:
        return {'error': f'未知股票 {stock}，可选 {TICKERS}'}
    r = _RETS[stock]
    return {'年化收益': round(float(annualized_return(r)), 4),
            '年化波动': round(float(annualized_volatility(r)), 4),
            '夏普': round(float(sharpe_ratio(r, 0.02)), 3),
            '最大回撤': round(float(max_drawdown(r)), 3)}

TOOLS = {'list_stocks': tool_list_stocks, 'metrics': tool_metrics}
print('可用工具:', list(TOOLS))
print('示例 metrics(TECH):', tool_metrics('TECH'))

## 2. ReAct 智能体循环

**ReAct = Reasoning + Acting**：智能体在「思考 → 行动（调用工具）→ 观察结果」之间循环，直到给出最终答复。
下面这二十来行就是一个**通用的 agent 主循环**——它本身与具体模型无关，规划器可换成规则或真实 LLM。

In [ ]:
def run_agent(query: str, planner, tools: dict, max_steps: int = 6, verbose: bool = True):
    """通用 ReAct 主循环。planner 接收 (query, history) 返回动作字典：
    {'thought':.., 'tool':名, 'args':{..}} 调用工具，或 {'thought':.., 'final':答复} 结束。
    """
    history = []  # [(action, observation), ...]
    for step in range(1, max_steps + 1):
        action = planner(query, history)
        if verbose:
            print(f'[步骤{step}] 思考：' + action.get('thought', ''))
        if 'final' in action:
            if verbose:
                print('[最终答复] ' + action['final'])
            return action['final'], history
        tool, args = action.get('tool'), action.get('args', {})
        obs = tools[tool](**args) if tool in tools else f'错误：未知工具 {tool}'
        if verbose:
            print(f'         行动：{tool}({args}) → 观察：{obs}')
        history.append((action, obs))
    return '达到最大步数仍未完成', history

## 3. 本地规则规划器（离线演示）

先用一个**规则规划器**代替 LLM，把 ReAct 的机制讲清楚：识别问题里提到的股票 → 逐个查指标 → 汇总作答。

In [ ]:
def rule_planner(query: str, history: list) -> dict:
    mentioned = [t for t in TICKERS if t in query]
    queried = [a['args']['stock'] for a, _ in history if a.get('tool') == 'metrics']
    todo = [t for t in mentioned if t not in queried]
    if todo:
        return {'thought': f'还需查询 {todo[0]} 的风险指标', 'tool': 'metrics', 'args': {'stock': todo[0]}}
    data = {a['args']['stock']: obs for a, obs in history if a.get('tool') == 'metrics'}
    if len(data) >= 2:
        names = list(data)
        safer = min(names, key=lambda s: data[s]['年化波动'])
        parts = [f"{s} 夏普 {data[s]['夏普']}、波动 {data[s]['年化波动']}" for s in names]
        final = '；'.join(parts) + f'。风险更低的是 {safer}。'
    else:
        final = f'指标：{data}'
    return {'thought': '指标已收集齐，可以作答', 'final': final}

answer, traj = run_agent('请对比 TECH 和 BANK 的风险', rule_planner, TOOLS)

## 4. 工具调用的结构化输出（JSON）

真实 LLM 通过**结构化输出（JSON）**告诉程序「调用哪个工具、参数是什么」。下面演示解析 LLM 风格的 JSON 动作并做健壮处理。

In [ ]:
def parse_action(text: str) -> dict:
    """从 LLM 文本中解析 JSON 动作；失败则回退为直接答复。"""
    try:
        start, end = text.index('{'), text.rindex('}') + 1
        return json.loads(text[start:end])
    except (ValueError, json.JSONDecodeError):
        return {'thought': '解析失败', 'final': text.strip()}

llm_output = '我先查指标。{"thought": "查询TECH", "tool": "metrics", "args": {"stock": "TECH"}}'
print('解析结果:', parse_action(llm_output))
print('非JSON回退:', parse_action('直接回答：风险较高。'))

## 5. 多智能体协作（研究 → 批判 → 综合）

复杂任务可由**多个智能体分工**：研究员给初稿、批判者挑问题、综合者定稿，体现协作流水线。

In [ ]:
def researcher(stock: str) -> str:
    m = tool_metrics(stock)
    return f"{stock} 年化收益 {m['年化收益']}、夏普 {m['夏普']}，看起来值得关注。"

def critic(draft: str, stock: str) -> str:
    m = tool_metrics(stock)
    if m['最大回撤'] < -0.3:
        return f"批判：初稿忽略风险——{stock} 最大回撤达 {m['最大回撤']:.0%}，回撤过深。"
    return '批判：未发现重大遗漏。'

def synthesizer(draft: str, critique: str) -> str:
    return f'综合结论：{draft} 但需注意：{critique}'

stock = 'TECH'
draft = researcher(stock)
crit = critic(draft, stock)
print('研究员：', draft)
print('批判者：', crit)
print('综合者：', synthesizer(draft, crit))

## 6. Agentic RAG：把检索当作一个工具

把第19章的 RAG **包装成一个工具**，智能体就能在需要时主动检索知识库——这就是 agentic RAG。

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

KB = ['银行股通常低波动、防御性强',
      '科技股波动大、成长性高、回撤可能很深',
      '白酒股属于消费板块，历史上高成长',
      '公用事业股最稳健，适合保守配置']
_vec = TfidfVectorizer(tokenizer=list, token_pattern=None).fit(KB)
_mat = _vec.transform(KB)

def tool_retrieve(query: str, k: int = 2) -> list:
    """从知识库检索最相关的 k 条。"""
    sims = cosine_similarity(_vec.transform([query]), _mat)[0]
    return [KB[i] for i in sims.argsort()[::-1][:k]]

TOOLS['retrieve'] = tool_retrieve
print('检索「科技股风险」:', tool_retrieve('科技股风险'))

## 7. 风险护栏：禁止智能体自主下单

!!! danger 金融场景的红线
    **绝不能让智能体自主执行交易/下单等高危操作。** 这里演示**工具白名单 + 人在回路（HITL）**护栏：
    危险工具必须经人工批准才能执行。

In [ ]:
SAFE_TOOLS = {'list_stocks', 'metrics', 'retrieve'}   # 只读工具，允许自主调用
DANGEROUS_TOOLS = {'place_order'}                      # 高危工具，必须人工批准

def guarded_execute(tool: str, args: dict, human_approved: bool = False):
    if tool in DANGEROUS_TOOLS and not human_approved:
        return f'⛔ 已拦截：{tool} 属高危操作，需人工批准（HITL）后才能执行'
    if tool in SAFE_TOOLS:
        return TOOLS[tool](**args)
    return f'⛔ 未授权工具：{tool}'

print(guarded_execute('metrics', {'stock': 'BANK'}))
print(guarded_execute('place_order', {'stock': 'TECH', 'qty': 100}))
print(guarded_execute('place_order', {'stock': 'TECH', 'qty': 100}, human_approved=True))

## 8. 真实 LLM 规划器（可选，guarded）

把规则规划器换成**真实 LLM**：用 OpenAI 兼容接口（可经 `base_url` 接国产模型）。
需 `uv sync --extra agent` 并设置 `OPENAI_API_KEY`（及可选 `OPENAI_BASE_URL`）。**未配置时自动回退到规则规划器**，离线可跑。

In [ ]:
import os

TOOL_SPEC = ('可用工具：list_stocks() 列出股票；metrics(stock) 返回风险指标。'
             '只输出一个 JSON：调用工具用 {"thought":..,"tool":..,"args":{..}}，'
             '作答用 {"thought":..,"final":..}')

def llm_planner(query: str, history: list) -> dict:
    try:
        from openai import OpenAI
        if not os.environ.get('OPENAI_API_KEY'):
            raise RuntimeError('未设置 OPENAI_API_KEY')
        client = OpenAI()  # 可经 OPENAI_BASE_URL 接国产模型
        msg = f'{TOOL_SPEC}\n\n问题：{query}\n已有观察：{[o for _, o in history]}'
        resp = client.chat.completions.create(
            model=os.environ.get('LLM_MODEL', 'gpt-4o-mini'),
            messages=[{'role': 'user', 'content': msg}], temperature=0)
        return parse_action(resp.choices[0].message.content)
    except Exception as e:
        print(f'（未用真实 LLM：{type(e).__name__}，回退规则规划器）')
        return rule_planner(query, history)

ans, _ = run_agent('对比 TECH 和 UTILITY 的风险', llm_planner, TOOLS)

## 习题参考解答

**习题1：给智能体新增工具 `cumulative_return(stock)`，并回答「哪只股票累计收益最高」。**

In [ ]:
def tool_cum_return(stock: str) -> float:
    if stock not in _RETS:
        return float('nan')
    return round(float((1 + _RETS[stock]).prod() - 1), 4)

TOOLS['cum_return'] = tool_cum_return
best = max(TICKERS, key=tool_cum_return)
print({t: tool_cum_return(t) for t in TICKERS})
print('累计收益最高:', best)

**习题2：验证规则规划器支持三只及以上股票的对比。**

In [ ]:
answer3, _ = run_agent('对比 BANK、TECH 和 UTILITY 的风险', rule_planner, TOOLS, verbose=False)
print(answer3)

**习题3：为护栏增加「任一工具调用超过 3 次即中止」，防止失控循环。**

In [ ]:
from collections import Counter

def run_agent_capped(query, planner, tools, max_steps=8, cap=3):
    history, calls = [], Counter()
    for _ in range(max_steps):
        action = planner(query, history)
        if 'final' in action:
            return action['final']
        calls[action['tool']] += 1
        if calls[action['tool']] > cap:
            return f'⛔ 工具 {action["tool"]} 调用超过 {cap} 次，已中止（防失控）'
        obs = tools[action['tool']](**action.get('args', {}))
        history.append((action, obs))
    return '达到最大步数'

print(run_agent_capped('对比 TECH 和 BANK 的风险', rule_planner, TOOLS))